In [1]:
import torch 
import torch.nn as nn
import torch.optim as optim
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
class PINN(nn.Module):
    def __init__(self):
        super(PINN, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(3, 64),
            nn.Tanh(),
            nn.Linear(64, 64),
            nn.Tanh(),
            nn.Linear(64, 1)
            )
        
    def forward(self, x):
        return self.net(x)

In [3]:
def initial_condition(x, y):
    return torch.sin(torch.pi * x) * torch.sin(torch.pi * y)

def boundary_condition(x, y, t, custom_value):
    return torch.full_like(x, custom_value)
    

In [4]:
def generate_training_data(num_points):
    x = torch.rand(num_points, 1, requires_grad=True)
    y = torch.rand(num_points, 1, requires_grad=True)
    t = torch.rand(num_points, 1, requires_grad=True)

    return x, y, t

In [6]:
def generate_boundary_points(num_points):
    x_boundary = torch.tensor([0.0, 1.0]).repeat(num_points//2)
    y_boundary = torch.rand(num_points)

    if torch.rand(1) > 0.5:
        x_boundary, y_boundary = y_boundary, x_boundary

    return x_boundary.view(-1,1), y_boundary.view(-1, 1)
    
def generate_boundary_training_data(num_points):
    x_boundary, y_boundary = generate_boundary_points(num_points=num_points)
    t = torch.rand(num_points, 1, requires_grad=True)
    return x_boundary, y_boundary, t

In [ ]:
def pde(x, y, t, model):
    input_data = torch.cat([x,y,t], dim=1)
    u = model(input_data)
    u_x, u_y = torch.autograd(u, [x,y], grad_outputs=torch.ones_like(u), create_graph=True, retain_graph=True)
    u_xx = torch.autograd(u_x, grad_outputs=torch.ones_like(u_x), create_graph=True, retain_graph=True)[0]
    u_yy = torch.autograd(u_y, grad_outputs=torch.ones_like(u_y), create_graph=True, retain_graph=True)[0]
    u_t = torch.autograd(u, t, grad_outputs=torch.ones_like(u), create_graph=True, retain_graph=True)[0]
    
    heat_eq_residual = 1 * u_xx + 1 * u_yy - u_t

    return heat_eq_residual

In [ ]:
def train_PINN(model, num_iterations, num_points):
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    mse = nn.MSELoss()

    for iteration in range(num_iterations):
        optimizer.zero_grad()

        x, y, t = generate_training_data(num_points=num_points)

        x_b, y_b, t_b = generate_boundary_training_data(num_points=num_points)

        t_initial = torch.zeros_like(t)
        u_initial = initial_condition(x, y)

        custom_value = 0
        u_b_boundary_x = boundary_condition(
            x_b, y_b, t_b, custom_value=custom_value
        )

        u_b_boundary_y = boundary_condition(
            y_b, x_b, t_b, custom_value=custom_value
        )

        u0_pred = model(torch.cat([x, y, t_initial], dim=1))
        ubx_pred = model(torch.cat([x_b, y_b, t_b], dim=1))
        uby_pred = model(torch.cat([y_b, x_b, t_b], dim=1))

        residual = pde(x, y, t, model=model)

        loss_ic = mse(u0_pred, u_initial)
        loss_bx = mse(ubx_pred, u_b_boundary_x)
        loss_by = mse(uby_pred, u_b_boundary_y)
        loss_pde = mse(residual, torch.zeros_like(residual))

        loss = loss_ic + loss_bx + loss_by + loss_pde

        loss.backward()
        optimizer.step()

        if iteration % 100 == 0:
            print(
                f"Iteration {iteration:6d} | "
                f"Loss = {loss.item():.6e} | "
                f"IC = {loss_ic.item():.6e} | "
                f"BX = {loss_bx.item():.6e} | "
                f"BY = {loss_by.item():.6e} | "
                f"PDE = {loss_pde.item():.6e}"
            )